In [0]:
# 1. Install dependency
%pip install pyyaml openpyxl


In [0]:
dbutils.library.restartPython()

In [0]:

# 2. Import libraries
import yaml
import pandas as pd

In [0]:
display(dbutils.fs.ls("/Volumes/PEI/raw/data/"))

In [0]:

# raw = (
#     spark.read.format("json")
#     .option("multiLine", "true")
#     .option("mode", "PERMISSIVE")
#     .load("dbfs:/Volumes/PEI/raw/data/Orders.json")
# )
# display(raw)
# # sink_path = "dbfs:/Volumes/pei/bronze/orders/"

# # print(f"Writing to: {sink_path}")

# # (
# #     raw.write
# #     .mode("overwrite")
# #     .format("parquet")
# #     .save(sink_path)
# # )

# # raw.show(2)

In [0]:


# # 3. Import your function (if modularized)
# # Option A: if using workspace files
# # from src.ingestion import read_data_sink

# # 4. Load config
# config_path = "/Workspace/Users/harkiratkr8@gmail.com/PEI_ProfitAnalysis/configs/bronze/orders_ingest.yaml"

# with open(config_path, "r") as f:
#     config = yaml.safe_load(f)

# # 5. Run pipeline
# for dataset in config["datasets"]:
#     read_data_sink(dataset)

In [0]:
# for dataset in config["datasets"]:
#     print(f"Processing: {dataset['name']}")
#     read_data_sink(dataset)

# CHANGES


In [0]:
def read_csv(spark, config):
    return (
        spark.read.format("csv")
        .options(**config.get("options", {}))
        .load(config["source_path"])
    )


def read_json(spark, config):
    return (
        spark.read.format("json")
        .options(**config.get("options", {}))
        .load(config["source_path"])
    )


# def read_xlsx(spark, config):
#     import pandas as pd
#     df_pd = pd.read_excel(config["source_path"])
#     return spark.createDataFrame(df_pd)

def read_xlsx(spark, config):
    import pandas as pd

    df_pd = pd.read_excel(config["source_path"])

    # Fix problematic columns
    if "phone" in df_pd.columns:
        df_pd["phone"] = df_pd["phone"].astype(str)

    return spark.createDataFrame(df_pd)

In [0]:
READERS = {
    "csv": read_csv,
    "json": read_json,
    "parquet": lambda spark, config: spark.read.parquet(config["source_path"]),
    "xlsx": read_xlsx
}


def get_reader(format):
    if format not in READERS:
        raise ValueError(f"Unsupported format: {format}")
    return READERS[format]

In [0]:
config_path = "/Workspace/Users/harkiratkr8@gmail.com/PEI_ProfitAnalysis/configs/bronze/orders_ingest.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

In [0]:
def ingest_dataset(config):

    reader_fn = get_reader(config["format"])

    df = reader_fn(spark, config)

    df = standardize(df, config)

    write_to_bronze(df, config)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

def standardize(df, config):
    return (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", lit(config["source_path"]))
        .withColumn("dataset_name", lit(config["name"]))
    )

In [0]:
def write_to_bronze(df, config):
    target_path = config["target_path"]

    (
        df.write
        .mode(config.get("write_mode", "overwrite"))
        .format("parquet")   # or delta in real world
        .save(target_path)
    )

In [0]:
for dataset in config["datasets"]:
    try:
        ingest_dataset(dataset)
    except Exception as e:
        print(f"Failed: {dataset['name']} → {str(e)}")